# 파서 워크벤치

**두 줄:** 파일 경로를 적으면 그 파일이 온다. 읽기·가공을 자유폼 셀에 쓰고, 마지막 셀이 파서 파일을 만든다.

> 🔴 **이 노트북은 운영 코드를 «부른다». 어떤 단계도 다시 구현하지 않는다.**
> 읽기·클레임 훑기·세 단계 분리·발행 대조는 전부 `dev_bench` 의 함수이고, 그 함수들이 부르는 것은
> `pipeline_base` · `directory_watcher` 다. 여기서 되는 것이 운영에서 안 되면 도구가 신뢰받는 바로
> 그 순간에 거짓말을 한 것이다.

**여는 법:** VS Code/Cursor 노트북 편집기 + 커널 conda `assy_manager`.
이 박스에 `jupyter notebook`/`jupyterlab` 은 없다(`ipykernel`·`jupyter_client` 는 있다).

## 0. 부트스트랩

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# 저장소 뿌리를 «찾는다» — 이 박스의 절대경로를 박지 않는다.
NB_DIR = Path.cwd()
REPO = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "server" / "dev_bench.py").is_file())
SERVER = REPO / "server"
if str(SERVER) not in sys.path:
    sys.path.insert(0, str(SERVER))

import pandas as pd

import dev_bench

# 🔴 커널을 확인한다. env 가 아닌 python 으로 돌면 psycopg2 가 없어 DB 셀이 «조용히» 못 연다.
print("interpreter :", sys.executable)
print("repo        :", REPO)
print("pandas      :", pd.__version__)

from parsers import directory_watcher

# 플러그인이 `import pipeline_base`(최상위 이름)로 베이스를 찾게 한다. 운영이 파서를 로드하기
# «직전»에 부르는 바로 그 함수다 — 흉내 내면 베이스가 두 정체성으로 로드돼 `issubclass` 가
# 깨지고, 로더가 «예외 없이» 「파서가 없다」고 답한다.
directory_watcher.prepare_plugin_imports()

## 1. 내가 고를 것 — **이 셀만 고친다**

In [ ]:
FILE  = ""                    # 개발할 파일 하나. 비우면 raws/ 에서 가장 최근 것
TABLE = "inventory_master"    # ingestion_workspace 폴더 이름 (= 발행될 scripts/ 의 주인)
NAME  = "my_parser"           # 발행할 모듈 이름. 클래스 이름은 여기서 만들어진다

WORKSPACE = SERVER / "ingestion_workspace" / TABLE
SCRIPTS, RAWS = WORKSPACE / "scripts", WORKSPACE / "raws"


def _newest(folder):
    files = [p for p in Path(folder).rglob("*") if p.is_file()]
    return max(files, key=lambda p: p.stat().st_mtime) if files else None


SAMPLE = Path(FILE) if FILE else (_newest(RAWS) if RAWS.is_dir() else None)
assert SAMPLE and SAMPLE.is_file(), f"FILE 에 경로를 적을 것 (raws={RAWS})"
print("sample  :", SAMPLE, f"({SAMPLE.stat().st_size:,} bytes)")
print("scripts :", SCRIPTS if SCRIPTS.is_dir() else f"{SCRIPTS}  [없음 — 발행이 만든다]")

## 2. 파일에 «무엇이 있나»

⚠️ **읽기 실패는 정상이다.** 운영의 기본 읽기(`pd.read_csv`/`read_excel`)가 못 여는 포맷이
아래 3번 셀이 있는 이유다. 실패하면 `read_refusal` 에 사유가 «값으로» 오고, 바이트와 줄은
그대로 보인다 — 읽기를 «쓰려면» 그게 필요하다.

In [ ]:
RAW = dev_bench.raw_for_parser(str(SAMPLE))

print("encoding :", RAW["encoding"], "  bytes:", f"{RAW['size_bytes']:,}")
print("read     :", RAW["read_refusal"] or "운영 기본 읽기 OK")
for line in RAW["lines"][:10]:
    print("   ", line[:120])

RAW["df"].head() if RAW["df"] is not None else None

## 3. 읽기 — **자유폼**

인자 이름이 `file_path` 인 것은 우연이 아니다: 이 본문은 그대로
`BasePipelineParser._read_file_to_dataframe(self, file_path)` 의 본문이 된다.
운영 기본 읽기가 되는 파일이면 `RAW["df"]` 를 그대로 쓰면 된다 — 그때는 이 셀이 없어도 되고,
`None` 을 돌려주면 베이스의 읽기가 발행 파일에 남는다.

In [ ]:
def read_df(file_path):
    import pandas as pd

    return pd.read_csv(file_path)


DF = read_df(str(SAMPLE))
DF.head()

## 4. 가공 — **자유폼**

여기 본문이 `process_dataframe(self, df)` 의 본문이 된다. 컬럼 이름·타입만 여기서 —
읽기 규칙은 위 셀이다.

In [ ]:
def process(df):
    return df


OUT = process(DF)
print(f"{len(DF):,} rows x {len(DF.columns)} cols  ->  {len(OUT):,} x {len(OUT.columns)}")
OUT.head()

## 5. 이 파일을 «누가» 집나

운영은 `match()` 가 True 인 **첫 번째** 파서를 쓴다. 그래서 「둘이 집는 상태」는 「하나가 집는
상태」와 똑같아 보이고, 남의 행이 내 표에 앉기 전까지 안 보인다. 아래는 아무것도 claim 하지 않고
**전부** 묻는다.

In [ ]:
survey = (dev_bench.claimers(str(SAMPLE), scripts_path=str(SCRIPTS))
          if SCRIPTS.is_dir() else {"rows": [], "winners": [], "load_errors": {}})

for row in survey["rows"]:
    mark = {True: "[집는다]", False: "[  -  ]", None: "[예외!]"}[row["match"]]
    print(mark, f"{row['script']}::{row['class']}", row["error"] or "")
for name, why in survey["load_errors"].items():
    print("[로드 실패]", name, why.strip().splitlines()[-1])

print("\n집는 것:", [r["class"] for r in survey["winners"]] or "없음 (std 파서 폴백 대상)")

# 이미 집는 파서가 있으면 그 «세 단계»를 따로 본다 — 무엇이 내 가정과 갈리는지는
# read / process / clean 이 갈려 있어야 보인다. parse() 는 「됐다/안 됐다」만 답한다.
if survey["winners"]:
    S = dev_bench.run_stages(survey["winners"][0]["cls"], str(SAMPLE))
    print(f"  1 raw       {S.raw.shape[0]:,} x {S.raw.shape[1]}")
    print(f"  2 processed {S.processed.shape[0]:,} x {S.processed.shape[1]}")
    print(f"  3 records   {len(S.records):,} dicts")

## 6. 발행 — 그리고 **운영 경로로 다시 채점한다**

🔴 옮겨 적지 않는다. `cell_body()` 가 위 두 셀의 본문을 그대로 들고 가고, `publish_parser` 는
파일을 쓴 «뒤» 같은 파일을 **클레임 → parse** 로 다시 돌려 위 `OUT` 과 대조한다. 다르면 파일을
지우고 이름을 대어 거절한다 — 그래서 「노트북에서 됐다」와 「배포된 것」이 정의상 같다.

⚠️ `match()` 는 **초안**이다(기본 = 이 파일의 확장자). 운영이 읽는 폴더에 앉기 «전»에 좁힐 것.

In [ ]:
published = dev_bench.publish_parser(
    NAME,
    dev_bench.cell_body(read_df),
    dev_bench.cell_body(process),
    file=str(SAMPLE),
    expected=OUT,
    match_pattern=None,
    scripts_path=str(SCRIPTS),
)
print("작성   :", published["path"])
print("클레임 :", published["who"], "->", f"{len(published['rows']):,} records")